In [ ]:
import cv2
import json
import numpy as np
from collections import defaultdict
from insightface.app import FaceAnalysis


class FaceTracker:
    def __init__(self, similarity_threshold=0.65, max_disappeared=30, update_alpha=0.9):
        self.next_person_id = 1
        self.active_people = {}
        self.similarity_threshold = float(similarity_threshold)
        self.max_disappeared = int(max_disappeared)
        self.update_alpha = float(update_alpha)
        self.disappeared_frames = defaultdict(int)

    def _normalize(self, emb):
        emb = np.asarray(emb, dtype=np.float32).reshape(-1)
        norm = float(np.linalg.norm(emb) + 1e-12)
        return emb / norm

    def get_cosine_similarity(self, embedding1, embedding2):
        e1 = self._normalize(embedding1)
        e2 = self._normalize(embedding2)
        return float(np.dot(e1, e2))

    def _register(self, embedding, timestamp):
        person_id = self.next_person_id
        self.active_people[person_id] = {
            "embedding": self._normalize(embedding),
            "last_seen": float(timestamp),
        }
        self.disappeared_frames[person_id] = 0
        self.next_person_id += 1
        return person_id

    def _mark_disappeared(self):
        for person_id in list(self.active_people.keys()):
            self.disappeared_frames[person_id] += 1
            if self.disappeared_frames[person_id] > self.max_disappeared:
                del self.active_people[person_id]
                del self.disappeared_frames[person_id]

    def update(self, faces, timestamp):
        if not faces:
            self._mark_disappeared()
            return []

        if not self.active_people:
            assigned = []
            for face_data in faces:
                assigned.append(self._register(face_data["embedding"], timestamp))
            return assigned

        person_ids = list(self.active_people.keys())
        stored = np.stack([self.active_people[pid]["embedding"] for pid in person_ids], axis=0)

        new_embeddings = [self._normalize(f["embedding"]) for f in faces]
        new_mat = np.stack(new_embeddings, axis=0)

        sim = new_mat @ stored.T

        assigned_ids = [None] * len(faces)
        used_people = set()
        used_faces = set()

        candidates = []
        for i in range(sim.shape[0]):
            for j in range(sim.shape[1]):
                candidates.append((float(sim[i, j]), i, j))
        candidates.sort(reverse=True, key=lambda x: x[0])

        for s, i, j in candidates:
            if s < self.similarity_threshold:
                break
            if i in used_faces:
                continue
            pid = person_ids[j]
            if pid in used_people:
                continue

            assigned_ids[i] = pid
            used_faces.add(i)
            used_people.add(pid)

            old_emb = self.active_people[pid]["embedding"]
            new_emb = new_embeddings[i]
            updated = self.update_alpha * old_emb + (1.0 - self.update_alpha) * new_emb
            self.active_people[pid]["embedding"] = self._normalize(updated)
            self.active_people[pid]["last_seen"] = float(timestamp)
            self.disappeared_frames[pid] = 0

        for i, face_data in enumerate(faces):
            if assigned_ids[i] is None:
                assigned_ids[i] = self._register(face_data["embedding"], timestamp)

        for pid in list(self.active_people.keys()):
            if pid not in used_people:
                self.disappeared_frames[pid] += 1
                if self.disappeared_frames[pid] > self.max_disappeared:
                    del self.active_people[pid]
                    del self.disappeared_frames[pid]

        return assigned_ids


def _bbox_xywh_from_insightface(bbox, frame_width, frame_height):
    x1, y1, x2, y2 = bbox
    x1 = int(round(float(x1)))
    y1 = int(round(float(y1)))
    x2 = int(round(float(x2)))
    y2 = int(round(float(y2)))

    x1 = max(0, min(x1, frame_width - 1))
    y1 = max(0, min(y1, frame_height - 1))
    x2 = max(0, min(x2, frame_width - 1))
    y2 = max(0, min(y2, frame_height - 1))

    w = max(0, x2 - x1)
    h = max(0, y2 - y1)
    return int(x1), int(y1), int(w), int(h)


def process_video(video_path, output_json, similarity_threshold=0.65, max_disappeared=30):
    print(f"Processing {video_path}...")

    # Check if CUDA is available
    import onnxruntime as ort
    available_providers = ort.get_available_providers()

    # Determine providers to use, prioritizing CUDA if available
    providers_to_use = []
    if "CUDAExecutionProvider" in available_providers:
        providers_to_use.append("CUDAExecutionProvider")
        print("✓ T4 GPU detected, using CUDA acceleration")
    elif "AzureExecutionProvider" in available_providers:
        providers_to_use.append("AzureExecutionProvider")
        print("Using AzureExecutionProvider.")
    else:
        providers_to_use.append("CPUExecutionProvider")
        print("Using CPUExecutionProvider.")

    app = FaceAnalysis(name="buffalo_s", providers=providers_to_use)
    app.prepare(ctx_id=0, det_size=(640, 640))

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    # Lower threshold prevents ID switching when faces turn slightly
    tracker = FaceTracker(similarity_threshold=0.45, max_disappeared=50)
    all_detections = []
    frame_count = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_count += 1
        timestamp = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0

        try:
            faces = app.get(frame)
            frame_h, frame_w = frame.shape[:2]

            face_items = []
            for f in faces:
                if getattr(f, "embedding", None) is None:
                    continue
                x, y, w, h = _bbox_xywh_from_insightface(f.bbox, frame_w, frame_h)
                if w <= 0 or h <= 0:
                    continue
                face_items.append({
                    "embedding": np.asarray(f.embedding, dtype=np.float32),
                    "bbox": [int(x), int(y), int(w), int(h)],
                })

            assigned_ids = tracker.update(face_items, timestamp)

            # Verbose logging for every frame
            ids_str = ", ".join([f"person_{pid}" for pid in assigned_ids]) if assigned_ids else "none"
            print(f"[Frame {frame_count:04d}] time={timestamp:.2f}s | faces_detected={len(face_items)} | assigned_ids=[{ids_str}] | total_active={len(tracker.active_people)})")

            for face, person_id in zip(face_items, assigned_ids):
                if person_id is None:
                    continue
                detection = {
                    "timestamp": round(float(timestamp), 2),
                    "person_id": f"person_{int(person_id)}",
                    "coordinates_pixels": [int(v) for v in face["bbox"]],
                }
                all_detections.append(detection)

        except Exception as e:
            print(f"Error processing frame {frame_count}: {str(e)}")
            continue

    cap.release()

    with open(output_json, "w") as f:
        json.dump(all_detections, f, indent=4)

    print(f"\nProcessing complete! Saved {len(all_detections)} detections to {output_json}")


def create_unique_color(identifier):
    hash_val = hash(identifier)
    r = (hash_val & 0xFF0000) >> 16
    g = (hash_val & 0x00FF00) >> 8
    b = hash_val & 0x0000FF
    return (b, g, r)


def visualize_tracking(input_video, input_json, output_video):
    print(f"Loading data from {input_json}...")
    with open(input_json, "r") as f:
        all_detections = json.load(f)

    detections_map = {}
    for item in all_detections:
        ts_key = round(float(item["timestamp"]), 2)
        if ts_key not in detections_map:
            detections_map[ts_key] = []
        detections_map[ts_key].append(item)

    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {input_video}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    print(f"Generating debug video: {output_video}...")
    frame_idx = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        current_ts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
        ts_key = round(float(current_ts), 2)

        current_faces = []
        keys_to_check = [
            ts_key,
            round(ts_key - 0.03, 2),
            round(ts_key + 0.03, 2),
        ]
        for k in keys_to_check:
            if k in detections_map:
                current_faces = detections_map[k]
                break

        if current_faces:
            for face in current_faces:
                pid = face["person_id"]
                x, y, w, h = face["coordinates_pixels"]

                color = create_unique_color(pid)
                cv2.rectangle(frame, (int(x), int(y)), (int(x + w), int(y + h)), color, 2)

                label = pid
                font_scale = 0.6
                thickness = 2
                font = cv2.FONT_HERSHEY_SIMPLEX

                (text_width, text_height), baseline = cv2.getTextSize(label, font, font_scale, thickness)

                cv2.rectangle(
                    frame,
                    (int(x), int(y - text_height - 10)),
                    (int(x + text_width + 10), int(y)),
                    color,
                    -1,
                )

                cv2.putText(
                    frame,
                    label,
                    (int(x + 5), int(y - 5)),
                    font,
                    font_scale,
                    (255, 255, 255),
                    thickness,
                )

        cv2.putText(
            frame,
            f"Frame: {frame_idx}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2,
        )

        out.write(frame)
        frame_idx += 1

        if frame_idx % 100 == 0:
            print(f"Processed {frame_idx} frames...")

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    print(f"Done! Open '{output_video}' to see the result.")


if __name__ == "__main__":
    video_path = "/content/data.mp4"
    output_json = "face_output.json"
    output_video = "meeting_debug_output.mp4"

    process_video(video_path, output_json)
    visualize_tracking(video_path, output_json, output_video)


Processing /content/data.mp4...
Using AzureExecutionProvider.
Applied providers: ['AzureExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'AzureExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['AzureExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'AzureExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['AzureExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'AzureExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['AzureExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'AzureExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_s/

In [ ]:
!pip install -U insightface onnxruntime-gpu onnxruntime deepface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 4.5 MB/s eta 0:00:00


In [ ]:
from insightface.app import FaceAnalysis
import numpy as np
from collections import defaultdict

In [ ]:
import cv2
import json
import numpy as np
from collections import defaultdict
from deepface import DeepFace


# Emotion smoothing settings
EMOTION_WINDOW_SECONDS = 5.0  # Time window to accumulate emotion scores
MIN_CONFIDENCE_THRESHOLD = 30.0  # Minimum confidence to consider an emotion


def create_unique_color(identifier):
    """Create a unique color based on the identifier."""
    hash_val = hash(identifier)
    r = (hash_val & 0xFF0000) >> 16
    g = (hash_val & 0x00FF00) >> 8
    b = hash_val & 0x0000FF
    return (b, g, r)


def get_emotion_color(emotion):
    """Return color based on emotion type."""
    emotion_colors = {
        "angry": (0, 0, 255),      # Red
        "disgust": (0, 128, 0),    # Dark Green
        "fear": (128, 0, 128),     # Purple
        "happy": (0, 255, 255),    # Yellow
        "sad": (255, 0, 0),        # Blue
        "surprise": (0, 165, 255), # Orange
        "neutral": (128, 128, 128) # Gray
    }
    return emotion_colors.get(emotion.lower(), (255, 255, 255))


def analyze_emotions(input_video, input_json, output_json, output_video):
    """
    Analyze emotions for each detected face using DeepFace.
    Uses face detections from InsightFace output JSON.
    """
    print(f"Loading face detections from {input_json}...")
    with open(input_json, "r") as f:
        all_detections = json.load(f)

    # Organize detections by timestamp
    detections_map = {}
    for item in all_detections:
        ts_key = round(float(item["timestamp"]), 2)
        if ts_key not in detections_map:
            detections_map[ts_key] = []
        detections_map[ts_key].append(item)

    # Open video
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {input_video}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Create video writer
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    print(f"Processing {total_frames} frames for emotion detection...")
    print(f"Output video: {output_video}")
    print(f"Emotion smoothing: {EMOTION_WINDOW_SECONDS}s window (prevents rapid switching)")

    all_emotion_detections = []
    frame_idx = 0

    # Emotion history for smoothing: {person_id: [(timestamp, emotion_scores), ...]}
    emotion_history = defaultdict(list)
    # Current stable emotion for each person: {person_id: (emotion, confidence)}
    stable_emotions = {}

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        frame_idx += 1
        current_ts = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0
        ts_key = round(float(current_ts), 2)

        # Find detections for this frame (with small tolerance)
        current_faces = []
        keys_to_check = [
            ts_key,
            round(ts_key - 0.03, 2),
            round(ts_key + 0.03, 2),
        ]
        for k in keys_to_check:
            if k in detections_map:
                current_faces = detections_map[k]
                break

        # Process each detected face
        for face in current_faces:
            pid = face["person_id"]
            x, y, w, h = face["coordinates_pixels"]

            # Ensure coordinates are within frame bounds
            x = max(0, int(x))
            y = max(0, int(y))
            w = min(int(w), width - x)
            h = min(int(h), height - y)

            if w <= 0 or h <= 0:
                continue

            # Extract face region
            face_roi = frame[y:y+h, x:x+w]

            if face_roi.size == 0:
                continue

            # Analyze emotion using DeepFace
            try:
                # Use enforce_detection=False since we already have detected faces
                result = DeepFace.analyze(
                    face_roi,
                    actions=["emotion"],
                    enforce_detection=False,
                    silent=True
                )

                # DeepFace returns a list, get first result
                if isinstance(result, list):
                    result = result[0]

                raw_emotion = result["dominant_emotion"]
                emotion_scores = result["emotion"]

                # Add to emotion history for this person
                emotion_history[pid].append((current_ts, emotion_scores))

                # Remove old entries outside the time window
                cutoff_time = current_ts - EMOTION_WINDOW_SECONDS
                emotion_history[pid] = [(ts, scores) for ts, scores in emotion_history[pid] if ts >= cutoff_time]

                # Calculate accumulated emotion scores over the time window
                accumulated_scores = defaultdict(float)
                for ts, scores in emotion_history[pid]:
                    for emotion, score in scores.items():
                        accumulated_scores[emotion] += score

                # Find the emotion with highest accumulated confidence
                dominant_emotion = max(accumulated_scores, key=accumulated_scores.get)
                confidence = emotion_scores[dominant_emotion]  # Use current frame's confidence for display

                # Store stable emotion
                stable_emotions[pid] = (dominant_emotion, confidence)

                # Save emotion detection
                emotion_detection = {
                    "timestamp": round(float(current_ts), 2),
                    "person_id": pid,
                    "coordinates_pixels": [int(x), int(y), int(w), int(h)],
                    "emotion": dominant_emotion,
                    "confidence": round(float(confidence), 2),
                    "all_emotions": {k: round(float(v), 2) for k, v in emotion_scores.items()}
                }
                all_emotion_detections.append(emotion_detection)

                # Draw on frame
                person_color = create_unique_color(pid)
                emotion_color = get_emotion_color(dominant_emotion)

                # Draw face rectangle with person color
                cv2.rectangle(frame, (x, y), (x + w, y + h), person_color, 2)

                # Draw person ID label
                label_pid = pid
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.5
                thickness = 2

                (text_w, text_h), _ = cv2.getTextSize(label_pid, font, font_scale, thickness)
                cv2.rectangle(frame, (x, y - text_h - 10), (x + text_w + 10, y), person_color, -1)
                cv2.putText(frame, label_pid, (x + 5, y - 5), font, font_scale, (255, 255, 255), thickness)

                # Draw emotion label below the box
                label_emotion = f"{dominant_emotion} ({confidence:.0f}%)"
                (emo_w, emo_h), _ = cv2.getTextSize(label_emotion, font, font_scale, thickness)
                cv2.rectangle(frame, (x, y + h), (x + emo_w + 10, y + h + emo_h + 10), emotion_color, -1)
                cv2.putText(frame, label_emotion, (x + 5, y + h + emo_h + 5), font, font_scale, (255, 255, 255), thickness)

            except Exception as e:
                # If emotion detection fails, just draw the box without emotion
                person_color = create_unique_color(pid)
                cv2.rectangle(frame, (x, y), (x + w, y + h), person_color, 2)
                cv2.putText(frame, pid, (x + 5, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, person_color, 2)

        # Add frame counter
        cv2.putText(
            frame,
            f"Frame: {frame_idx}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (0, 255, 0),
            2
        )

        out.write(frame)

        # Log progress every frame
        if current_faces:
            emotions_str = ", ".join([f"{d['person_id']}:{d.get('emotion', '?')}"
                                      for d in all_emotion_detections
                                      if d['timestamp'] == round(float(current_ts), 2)])
            print(f"[Frame {frame_idx:04d}] time={current_ts:.2f}s | faces={len(current_faces)} | emotions=[{emotions_str}]")
        else:
            print(f"[Frame {frame_idx:04d}] time={current_ts:.2f}s | faces=0")

    cap.release()
    out.release()
    cv2.destroyAllWindows()

    # Save emotion detections to JSON
    with open(output_json, "w") as f:
        json.dump(all_emotion_detections, f, indent=4)

    print(f"\n\u2713 Processing complete!")
    print(f"  - Emotion detections saved to: {output_json}")
    print(f"  - Annotated video saved to: {output_video}")
    print(f"  - Total emotion detections: {len(all_emotion_detections)}")


if __name__ == "__main__":
    input_video = "meeting_debug_output.mp4"
    input_json = "face_output.json"
    output_json = "emotion_detections.json"
    output_video = "meeting_emotions_output.mp4"

    analyze_emotions(input_video, input_json, output_json, output_video)

25-12-31 17:23:08 - Directory /root/.deepface has been created
25-12-31 17:23:08 - Directory /root/.deepface/weights has been created
Loading face detections from face_output.json...
Processing 555 frames for emotion detection...
Output video: meeting_emotions_output.mp4
Emotion smoothing: 5.0s window (prevents rapid switching)
[Frame 0001] time=0.00s | faces=0
[Frame 0002] time=0.04s | faces=0
[Frame 0003] time=0.08s | faces=0


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5
To: /root/.deepface/weights/facial_expression_model_weights.h5


25-12-31 17:23:09 - 🔗 facial_expression_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/facial_expression_model_weights.h5 to /root/.deepface/weights/facial_expression_model_weights.h5...


100%|██████████| 5.98M/5.98M [00:00<00:00, 113MB/s]


[Frame 0004] time=0.12s | faces=1 | emotions=[person_1:happy]
[Frame 0005] time=0.16s | faces=1 | emotions=[person_1:happy]
[Frame 0006] time=0.20s | faces=1 | emotions=[person_1:happy]
[Frame 0007] time=0.24s | faces=1 | emotions=[person_1:happy]
[Frame 0008] time=0.28s | faces=1 | emotions=[person_1:happy]
[Frame 0009] time=0.32s | faces=4 | emotions=[person_1:happy, person_2:fear, person_3:sad, person_4:sad]
[Frame 0010] time=0.36s | faces=6 | emotions=[person_5:angry, person_6:sad, person_1:happy, person_2:fear, person_3:sad, person_4:sad]
[Frame 0011] time=0.40s | faces=6 | emotions=[person_5:angry, person_6:happy, person_1:happy, person_2:fear, person_3:sad, person_4:sad]
[Frame 0012] time=0.44s | faces=6 | emotions=[person_5:angry, person_6:happy, person_1:happy, person_2:fear, person_3:sad, person_4:sad]
[Frame 0013] time=0.48s | faces=6 | emotions=[person_5:angry, person_6:happy, person_1:happy, person_2:fear, person_3:sad, person_4:sad]
[Frame 0014] time=0.52s | faces=6 | emo